In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import CosineDecay
import time
import gc
import csv

print("--- STARTING KAGGLE BATCH EXECUTION: STANDARD CONVLSTM ---")

# ==========================================
# PART 1: CONFIGURATION & DATA PIPELINE
# ==========================================
DATASET_PATH = "/kaggle/input/datasets/yashaswi15/aerosense-training-data/Delhi_NCR_Master_DataCube_Scaled.npy"
BATCH_SIZE = 8

print(f"Loading 6-Channel DataCube from: {DATASET_PATH}")
try:
    master_data = np.load(DATASET_PATH, mmap_mode='r')
except FileNotFoundError:
    print("Path adjusted. Trying Kaggle's shortened directory...")
    DATASET_PATH = "/kaggle/input/aerosense-training-data/Delhi_NCR_Master_DataCube_Scaled.npy"
    master_data = np.load(DATASET_PATH, mmap_mode='r')

total_days = master_data.shape[0]
train_split = int(total_days * 0.8) 

class SpatiotemporalGenerator(tf.keras.utils.Sequence):
    def __init__(self, data_array, start_idx, end_idx, input_window=7, batch_size=8, is_training=True):
        self.data = data_array
        self.start_idx = start_idx
        self.end_idx = end_idx - input_window
        self.input_window = input_window
        self.batch_size = batch_size
        self.is_training = is_training
        self.valid_indices = np.arange(self.start_idx, self.end_idx)
        if self.is_training:
            np.random.shuffle(self.valid_indices)

    def __len__(self):
        return int(np.ceil(len(self.valid_indices) / self.batch_size))

    def __getitem__(self, index):
        batch_idx = self.valid_indices[index * self.batch_size:(index + 1) * self.batch_size]
        X_batch, y_batch = [], []
        for i in batch_idx:
            X_batch.append(self.data[i : i + self.input_window])
            y_batch.append(self.data[i + self.input_window])
            
        X_batch = np.array(X_batch, dtype=np.float32)
        y_batch = np.array(y_batch, dtype=np.float32)
        
        # Generator-level safety net
        X_batch = np.nan_to_num(X_batch, nan=0.0, posinf=1.0, neginf=0.0)
        y_batch = np.nan_to_num(y_batch, nan=0.0, posinf=1.0, neginf=0.0)
        
        return X_batch, y_batch
    
    def on_epoch_end(self):
        if self.is_training:
            np.random.shuffle(self.valid_indices)

print("Initializing Training and Validation Generators...")
train_gen = SpatiotemporalGenerator(master_data, 0, train_split, batch_size=BATCH_SIZE, is_training=True)
val_gen   = SpatiotemporalGenerator(master_data, train_split, total_days, batch_size=BATCH_SIZE, is_training=False)


# ==========================================
# PART 2: STANDARD CONVLSTM ARCHITECTURE
# ==========================================
print("\n[2/3] Building Spatial-Temporal Neural Network...")

def build_standard_convlstm(input_shape=(7, 141, 231, 6)):
    inputs = Input(shape=input_shape)

    x = layers.ConvLSTM2D(filters=16, kernel_size=(3, 3), padding='same', return_sequences=True, data_format='channels_last')(inputs)
    x = layers.BatchNormalization()(x)

    x = layers.ConvLSTM2D(filters=32, kernel_size=(3, 3), padding='same', return_sequences=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(filters=32, kernel_size=(3, 3), activation='relu', padding='same')(x)
    # Outputting all 6 channels
    outputs = layers.Conv2D(filters=6, kernel_size=(1, 1), activation='sigmoid', padding='same')(x)

    return models.Model(inputs=inputs, outputs=outputs, name="Standard_ConvLSTM_6Ch")

model = build_standard_convlstm()


# ==========================================
# PART 3: ADVANCED CUSTOM TRAINING LOOP (WITH NATIVE CSV LOGGER)
# ==========================================
MODEL_NAME = "Standard_ConvLSTM" 
MODEL_SAVE_PATH = f"/kaggle/working/Delhi_NCR_{MODEL_NAME}_Best.keras"
CSV_LOG_PATH = f"/kaggle/working/{MODEL_NAME}_Training_Log.csv"

# Create the CSV file and write the headers
with open(CSV_LOG_PATH, mode='w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "train_loss", "val_loss", "val_mape"])

print(f"\n[3/3] Initiating Custom Training Loop for {MODEL_NAME}...")
print(f" -> Metrics will be safely logged to: {CSV_LOG_PATH}")

initial_learning_rate = 0.001
decay_steps = 50 * len(train_gen)
lr_schedule = CosineDecay(initial_learning_rate, decay_steps)
optimizer = Adam(learning_rate=lr_schedule, clipnorm=1.0) # Crucial for LSTMs!

def calculate_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    
    mae = tf.reduce_mean(tf.abs(y_true - y_pred))
    # Using max_val=1.0 since target is sigmoided
    ssim = 1.0 - tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))
    return (0.5 * mae) + (0.5 * ssim)

@tf.function
def train_step(x_batch, y_batch):
    with tf.GradientTape() as tape:
        predictions = model(x_batch, training=True)
        loss = calculate_loss(y_batch, predictions)
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return loss

@tf.function
def val_step(x_batch, y_batch):
    predictions = model(x_batch, training=False)
    val_loss = calculate_loss(y_batch, predictions)
    
    # Mathematically calculate MAPE to save to the CSV
    y_true_f = tf.cast(y_batch, tf.float32)
    pred_f = tf.cast(predictions, tf.float32)
    mape = tf.reduce_mean(tf.abs((y_true_f - pred_f) / (y_true_f + 1e-10))) * 100.0
    
    return val_loss, mape

EPOCHS = 50
best_val_loss = float('inf')
patience = 7
patience_counter = 0

print(f"\n--- Starting 50-Epoch Backpropagation ---")

for epoch in range(EPOCHS):
    start_time = time.time()
    epoch_loss_avg = tf.keras.metrics.Mean()
    epoch_val_loss_avg = tf.keras.metrics.Mean()
    epoch_val_mape_avg = tf.keras.metrics.Mean() 
    
    for step in range(len(train_gen)):
        x_batch, y_batch = train_gen[step]
        loss_val = train_step(x_batch, y_batch)
        epoch_loss_avg.update_state(loss_val)
        
    for step in range(len(val_gen)):
        x_val, y_val = val_gen[step]
        v_loss, v_mape = val_step(x_val, y_val)
        epoch_val_loss_avg.update_state(v_loss)
        epoch_val_mape_avg.update_state(v_mape)
        
    train_loss = epoch_loss_avg.result().numpy()
    val_loss = epoch_val_loss_avg.result().numpy()
    val_mape = epoch_val_mape_avg.result().numpy()
    
    current_lr = optimizer.learning_rate(optimizer.iterations).numpy() if callable(optimizer.learning_rate) else optimizer.learning_rate.numpy()
        
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Time: {time.time() - start_time:.1f}s | LR: {current_lr:.5f} | Train Loss: {train_loss:.5f} | Val Loss: {val_loss:.5f} | Val MAPE: {val_mape:.2f}%")
    
    # APPEND METRICS TO NATIVE CSV FILE
    with open(CSV_LOG_PATH, mode='a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([epoch + 1, train_loss, val_loss, val_mape])
    
    if val_loss < best_val_loss:
        print(f"  -> Val Loss improved from {best_val_loss:.5f} to {val_loss:.5f}. Saving weights!")
        best_val_loss = val_loss
        model.save(MODEL_SAVE_PATH)
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"  -> No improvement. Patience: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print(f"\n[!] Early Stopping Triggered.")
            break
            
    # Force garbage collection to prevent Kaggle RAM crashes
    gc.collect()
    tf.keras.backend.clear_session()

print(f"\nTraining Complete! Logs successfully written to {CSV_LOG_PATH}")